# AI Detector, Probably - logo extraction (auto-discovery)Runs everything in this kernel. No CLI, no deployed functions, no hardcoded paths.**Before running:**1. Sidebar -> **Files panel** -> attach the two volumes that hold the checkpoint and the retrieval index (the cell finds them wherever they mount, but they must be attached).2. Sidebar -> **Compute profile**: GPU `A10G`, memory 16+ GB, idle timeout 60 min.3. Run cell 1 (installs packages), then cell 2. Wait for the final `ALL DONE` print (UMAP takes ~15-20 min).

In [ ]:
%uv pip install umap-learn sentence-transformers usearch

In [ ]:
# ============================================================# 1. LOCATE EVERYTHING (no assumptions about mount paths)# ============================================================import osfrom pathlib import Pathimport numpy as npimport pandas as pdimport torchimport matplotlib.pyplot as pltfrom datasets import load_datasetmnt = Path("/mnt")print("=== /mnt contents ===")if mnt.exists():    for p in sorted(mnt.iterdir()):        print("  ", p, "(dir)" if p.is_dir() else "")else:    print("  /mnt does not exist")# NOTE: rglob does NOT work on Modal volume mounts (returns empty even when# the files are there, as seen with iterdir). Use the known layout directly,# matching the eval scripts' defaults.ckpt_dir = Path("/mnt/weightsandotherstuff/pangram_final/pangram_best")if not ckpt_dir.exists():    ckpt_dir = Path("/mnt/weightsandotherstuff/pangram_best")index_path = Path("/mnt/dataset/ai_mirrors.usearch")ai_dir = Path("/mnt/dataset/ai_corpus")print("\ncheckpoint dir:", ckpt_dir, "(exists:", ckpt_dir.exists(), ")")print("index file:    ", index_path, "(exists:", index_path.exists(), ")")print("ai corpus dir: ", ai_dir, "(exists:", ai_dir.exists(), ")")if not ckpt_dir.exists():    raise SystemExit("checkpoint not found - expected /mnt/weightsandotherstuff/pangram_final/pangram_best. Attach the weightsandotherstuff volume (Files panel) and Restart kernel.")OUT_DIR = Path("/mnt/weightsandotherstuff/logo")OUT_DIR.mkdir(parents=True, exist_ok=True)print("output dir:    ", OUT_DIR)# ============================================================# 2. MODEL# ============================================================from transformers import DebertaV2ForSequenceClassification, DebertaV2TokenizerFastdevice = "cuda" if torch.cuda.is_available() else "cpu"print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")model = DebertaV2ForSequenceClassification.from_pretrained(str(ckpt_dir), num_labels=2)tokenizer = DebertaV2TokenizerFast.from_pretrained(str(ckpt_dir))model.eval().to(device)model.config.output_hidden_states = Trueprint("model loaded")# ============================================================# 3. DATA + HELPERS (held-out essay sources, same as eval_essays.py)# ============================================================BG = "#0d0d0d"; HUMAN_COLOR = "#4dabf7"; AI_COLOR = "#ffa94d"; MAX_LENGTH = 512BENCHMARK_SOURCES = {    "human": [        {"name": "HC3-Human", "text_field": "human_answers", "is_list_field": True,         "data_files": ["https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/refs%2Fconvert%2Fparquet/all/train/0000.parquet"]},        {"name": "Reddit-Writing", "text_field": "content", "is_list_field": False,         "filter_fn": lambda x: len(x.get("content", "").split()) > 150,         "data_files": [f"https://huggingface.co/datasets/webis/tldr-17/resolve/refs%2Fconvert%2Fparquet/default/partial-train/{i:04d}.parquet" for i in range(10)]},    ],    "ai": [        {"name": "HC3-ChatGPT", "text_field": "chatgpt_answers", "is_list_field": True,         "data_files": ["https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/refs%2Fconvert%2Fparquet/all/train/0000.parquet"]},        {"name": "GPT-Wiki-Intro", "text_field": "generated_intro", "is_list_field": False,         "data_files": ["https://huggingface.co/datasets/aadityaubhat/GPT-wiki-intro/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"]},    ],}def load_source(src, max_samples):    ds = load_dataset("parquet", data_files=src["data_files"], split="train", streaming=True)    texts, field, is_list = [], src["text_field"], src.get("is_list_field", False)    filt = src.get("filter_fn", lambda x: True)    for sample in ds:        if len(texts) >= max_samples:            break        if not filt(sample):            continue        if is_list:            answers = sample.get(field, [])            text = answers[0] if answers else ""        else:            text = sample.get(field, "")        if text and len(text.strip()) >= 100:            texts.append(text.strip())    print(f"   {src['name']}: {len(texts):,} samples")    return textsdef embed_cls(texts, batch_size=16):    embs, probs = [], []    bs = batch_size    i = 0    while i < len(texts):        chunk = texts[i:i+bs]        try:            inputs = tokenizer(chunk, truncation=True, max_length=MAX_LENGTH,                               padding=True, return_tensors="pt")            inputs = {k: v.to(device) for k, v in inputs.items()}            with torch.no_grad():                out = model(**inputs)                cls = out.hidden_states[-1][:, 0, :].float().cpu().numpy()                p = torch.softmax(out.logits, dim=-1)[:, 1].float().cpu().numpy()            embs.append(cls); probs.append(p)            i += len(chunk)        except torch.cuda.OutOfMemoryError:            if bs <= 1: raise            torch.cuda.empty_cache(); bs = max(1, bs // 2)    return np.concatenate(embs), np.concatenate(probs)def style(ax):    ax.set_facecolor(BG); ax.set_xticks([]); ax.set_yticks([])    for s in ax.spines.values(): s.set_visible(False)# ============================================================# 4. UMAP (the logo core)# ============================================================import umaptexts, labels, src_names = [], [], []for key, label in [("human", 0), ("ai", 1)]:    for src in BENCHMARK_SOURCES[key]:        t = load_source(src, 1500)        texts += t; labels += [label] * len(t); src_names += [src["name"]] * len(t)n_h = sum(1 for l in labels if l == 0)print(f"total: {len(texts):,} ({n_h:,} human, {len(labels)-n_h:,} AI)")assert 0 < n_h < len(labels), "single-class set"print("extracting [CLS] embeddings ...")emb, probs = embed_cls(texts)print("embeddings:", emb.shape)print("running UMAP ...")xy = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,               metric="cosine", random_state=42).fit_transform(emb)labels_arr = np.array(labels); human = labels_arr == 0fig, ax = plt.subplots(figsize=(9, 9), facecolor=BG); style(ax)ax.scatter(xy[human,0], xy[human,1], s=6, c=HUMAN_COLOR, alpha=0.65, linewidths=0, label="human")ax.scatter(xy[~human,0], xy[~human,1], s=6, c=AI_COLOR, alpha=0.65, linewidths=0, label="AI")ax.legend(facecolor="#1a1a1a", edgecolor="#333333", labelcolor="white", fontsize=13)fig.savefig(OUT_DIR / "umap_ground_truth.png", dpi=300, bbox_inches="tight", facecolor=BG)plt.show(); plt.close(fig)from matplotlib.colors import LinearSegmentedColormapcmap = LinearSegmentedColormap.from_list("h2a", [HUMAN_COLOR, "#f8f9fa", AI_COLOR])fig, ax = plt.subplots(figsize=(9, 9), facecolor=BG); style(ax)sc = ax.scatter(xy[:,0], xy[:,1], s=6, c=probs, cmap=cmap, vmin=0, vmax=1, linewidths=0)cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)cbar.set_label("P(AI)", color="white"); cbar.ax.tick_params(colors="white")cbar.outline.set_edgecolor("#333333")fig.savefig(OUT_DIR / "umap_confidence.png", dpi=300, bbox_inches="tight", facecolor=BG)plt.show(); plt.close(fig)pd.DataFrame({"source": src_names, "label": labels_arr, "ai_prob": probs,              "umap_x": xy[:,0], "umap_y": xy[:,1]}).to_csv(OUT_DIR / "umap_data.csv", index=False)print("UMAP saved:", sorted(p.name for p in OUT_DIR.iterdir()))# ============================================================# 5. MIRROR GRAPH (skips cleanly if the index volume is missing)# ============================================================if not index_path.exists():    print("\nWARNING: ai_mirrors.usearch not found - skipping mirror graph. Attach the dataset volume and re-run to get it.")else:    from usearch.index import Index    from sentence_transformers import SentenceTransformer    print("\nloading MiniLM index ...")    st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")    index = Index(ndim=st.get_sentence_embedding_dimension(), metric="cos", dtype="f16")    index.load(str(index_path))    parquet_files = sorted(str(p) for p in ai_dir.glob("*.parquet"))    print(f"corpus: {len(parquet_files)} parquet files")    ds = load_dataset("parquet", data_files=parquet_files, split="train")    n_humans, top_k = 60, 3    texts, src_names = [], []    for src in BENCHMARK_SOURCES["human"]:        t = load_source(src, n_humans)        texts += t; src_names += [src["name"]] * len(t)    texts, src_names = texts[:n_humans], src_names[:n_humans]    print(f"{len(texts)} human texts")    print("scoring humans with the detector ...")    _, probs = embed_cls(texts)    print("searching AI mirrors ...")    embs = st.encode(texts, convert_to_numpy=True, normalize_embeddings=True,                     batch_size=256, show_progress_bar=False)    matches = index.search(embs, top_k)    keys = np.atleast_2d(np.asarray(matches.keys))    dists = np.atleast_2d(np.asarray(matches.distances))    if keys.shape[0] == 1 and keys.shape[1] != top_k:        keys, dists = keys.T, dists.T    mirror_id_of_text, mirror_texts, pairs = {}, {}, []    for h in range(len(texts)):        for k_i in range(keys.shape[1]):            key = int(keys[h, k_i])            if key == -1: continue            try:                mtext = ds[key]["text"]            except (IndexError, KeyError):                continue            if mtext not in mirror_id_of_text:                mirror_id_of_text[mtext] = len(mirror_id_of_text)                mirror_texts[mirror_id_of_text[mtext]] = mtext            sim = 1.0 - float(dists[h, k_i])            pairs.append((h, mirror_id_of_text[mtext], sim))    print(f"{len(pairs)} pairs, {len(mirror_texts)} unique mirrors")    assert pairs, "no pairs - index/corpus mismatch?"    rng = np.random.default_rng(42)    order = np.argsort(probs, kind="stable")    human_y = {h: rank + rng.uniform(-0.25, 0.25) for rank, h in enumerate(order)}    mirror_y = {m: float(np.mean([human_y[h] for h, mm, _ in pairs if mm == m])) for m in mirror_texts}    mirror_ids = sorted(mirror_texts, key=lambda m: mirror_y[m])    fig, ax = plt.subplots(figsize=(12, 9), facecolor=BG); style(ax)    for h, m, sim in pairs:        ax.plot([0, 1], [human_y[h], mirror_y[m]], color="#888888", lw=0.6,                alpha=0.15 + 0.55 * sim, zorder=1)    ax.scatter([0.0]*len(texts), [human_y[h] for h in range(len(texts))],               s=[60 + 140 * float(probs[h]) for h in range(len(texts))],               c=HUMAN_COLOR, alpha=0.9, edgecolors="none", zorder=2)    ax.scatter([1.0]*len(mirror_ids), [mirror_y[m] for m in mirror_ids],               s=22, c=AI_COLOR, alpha=0.7, edgecolors="none", zorder=2)    fig.savefig(OUT_DIR / "mirror_graph.png", dpi=300, bbox_inches="tight", facecolor=BG)    plt.show(); plt.close(fig)    pd.DataFrame([{"human_source": src_names[h], "human_text": texts[h],                   "human_p_ai": float(probs[h]), "mirror_text": mirror_texts[m],                   "similarity": float(sim)} for h, m, sim in pairs]                 ).to_csv(OUT_DIR / "mirror_pairs.csv", index=False)    print("mirror graph saved:", sorted(p.name for p in OUT_DIR.iterdir()))print("\nALL DONE. Outputs in:", OUT_DIR)